In [2]:
import sys
sys.path.insert(0, '..')
from model.data import getMyData,getDataETTh1
from model.embedding import DataEmbedding
from model.encoder import EncoderInf, ConvLayer, EncoderInfLayer
from model.attention import AttentionLayer,ProbAttention

import torch_ep.embed as epEmbed
import torch_ep.encoder as epEncoder
import torch

import os
import re

import numpy  as np
import tensorflow as tf
import math
import pandas as pd

from tensorflow.keras.layers import Dense
from keras.backend import softmax

import matplotlib.pyplot as plt
import seaborn as sns
import seaborn.objects as so

In [ ]:
# Get Features from ETTh1
def getDataETTh1(batch_size, seq_len, global_size):
    root_path = "../../data/"
    data_path = "ETTh1.csv"
    df_raw = pd.read_csv(os.path.join(root_path,data_path))

    # split = re.compile('-|:| ')
    # featuresDate = (df_raw['date'].str.split(split, expand=True)).astype(np.int64).iloc[:,:4]
    featuresData = df_raw.iloc[:,1:8].astype(np.float32)

    # On ajoute les infos temporelles : mois/jour/joursemain/heure
    df_raw['mois'] = df_raw.date.apply(lambda row:int(row[5:7])).astype(np.int64)
    df_raw['jour'] = df_raw.date.apply(lambda row:int(row[8:10])).astype(np.int64)
    df_raw['heure'] = df_raw.date.apply(lambda row:int(row[11:13])).astype(np.int64)
    df_raw['sJour'] = df_raw['date'].astype('datetime64[s]').dt.dayofweek.astype(np.int64)
    featuresDate = df_raw[['mois','jour','sJour','heure']].astype(np.int64)
    # featuresDate.describe()

    # Convert Features to numpy array
    featuresData = featuresData.to_numpy()
    featuresDate = featuresDate.to_numpy()
    # Create a dataset Feature Data
    X = np.array([featuresData[i:i+seq_len] for i in range(0, featuresData.shape[0]-seq_len)], dtype=np.float32)
    Y = np.array([featuresData[i+seq_len] for i in range(0, featuresData.shape[0]-seq_len-1)], dtype=np.float32)
    Xt = tf.convert_to_tensor(X[:global_size,:,:], dtype=tf.float32)
    Yt = tf.convert_to_tensor(Y[:global_size,:], dtype=tf.float32)
    XT = torch.from_numpy(X[:global_size,:,:])

    # Create a dataset : Features Date
    X = np.array([featuresDate[i:i+seq_len] for i in range(0, featuresDate.shape[0]-seq_len)], dtype=np.float32)
    Y = np.array([featuresDate[i+seq_len] for i in range(0, featuresDate.shape[0]-seq_len-1)], dtype=np.float32)
    XtDate = tf.convert_to_tensor(X[:global_size,:,:], dtype=tf.float32)
    YtDate = tf.convert_to_tensor(Y[:global_size,:], dtype=tf.float32)
    XTDate = torch.from_numpy(X[:global_size,:])

    return (Xt,XtDate,Yt,YtDate,df_raw.iloc[0:global_size,:])

In [ ]:
# RECAP AttentionLayer
d_model = 512
HEAD = 8
RATE = 0.05
SEQ_LEN = 96
FACTOR = 5
BATCH_SIZE = 32
GLOBAL_SIZE = 32

tf.keras.backend.clear_session()

Xt, XtDate, Yt, YtDate, df_raw = getDataETTh1(batch_size=BATCH_SIZE, seq_len = SEQ_LEN, global_size = GLOBAL_SIZE)

#Embedding
encEmb = DataEmbedding(seq_len=SEQ_LEN, d_model=d_model,rate=RATE)
x = encEmb(x=Xt, x_mark=XtDate, training=True)

attLayer = AttentionLayer(ProbAttention(False,FACTOR,None,RATE,False),d_model=d_model,heads=HEAD)
newX, attn = attLayer(x,x,x,None)
Xt_min = tf.reduce_min(Xt)
Xt_max = tf.reduce_max(Xt)
fig = plt.figure(figsize=(18,2))
sns.lineplot(x=df_raw['date'][:GLOBAL_SIZE],y=df_raw['OT'][:GLOBAL_SIZE]).set_title('Data Brutes [0-32]');
fig = plt.figure(figsize=(18,2))
sns.heatmap(Xt[0,:,:], vmin=Xt_min, vmax=Xt_max, cmap=sns.color_palette("hls", 256)).set_title('Init Features Data');
fig = plt.figure(figsize=(18,2))
sns.heatmap(x[0,:,:], vmin=-10, vmax=10, cmap=sns.color_palette("hls", 256)).set_title('Embedding');
fig = plt.figure(figsize=(18,2))
sns.heatmap(newX[0,:,:], vmin=-10, vmax=10, cmap=sns.color_palette("hls", 256)).set_title('Attention');

In [ ]:
# Encoder
d_model = 512
HEAD = 8
RATE = 0.05
SEQ_LEN = 96
FACTOR = 5
BATCH_SIZE = 32
GLOBAL_SIZE = 1000

DEC_LEN = 72
ENC_LEN = 96

N = 2
d_FF= 2048

tf.keras.backend.clear_session()

x, x_date, Yt, YtDate, df_raw = getDataETTh1(batch_size=BATCH_SIZE, seq_len = SEQ_LEN, global_size = GLOBAL_SIZE)

start = np.random.randint(0,GLOBAL_SIZE-BATCH_SIZE)
x_enc, x_date_enc = [ x[start:start+BATCH_SIZE,:ENC_LEN,:] , x_date[start:start+BATCH_SIZE,:ENC_LEN,:]]
x_dec, x_date_dec = [ x[start:start+BATCH_SIZE,:DEC_LEN,:] , x_date[start:start+BATCH_SIZE,:DEC_LEN,:]]

#Embedding
EmbFct = DataEmbedding(seq_len=SEQ_LEN, d_model=d_model,rate=RATE,timeF=True)
encEmb = EmbFct(x=x_enc, x_mark=x_date_enc, training=True)
decEmb = EmbFct(x=x_dec, x_mark=x_date_dec, training =True)

e_layer = 2

encTest = EncoderInf( [ EncoderInfLayer( 
                        AttentionLayer(
                            ProbAttention(False,FACTOR,None,RATE,True)
                                ,d_model, HEAD, None, None, False)
                            ,d_model, RATE,d_FF) for _ in range(e_layer) 
                        ]                     ,
                    [ ConvLayer(seq_len=SEQ_LEN) for _ in range (e_layer - 1)
                    ])

out,attn = encTest(encEmb, training=True)
print(len(attn))

Xt_min = tf.reduce_min(x)
Xt_max = tf.reduce_max(x)
Out_min = tf.reduce_min(out)
Out_max = tf.reduce_max(out)
fig = plt.figure(figsize=(18,2))
sns.lineplot(x=df_raw['date'][:GLOBAL_SIZE],y=df_raw['OT'][:GLOBAL_SIZE]).set_title('Data Brutes [0-32]');
fig = plt.figure(figsize=(4,2))
sns.heatmap(attn[0][0,0,:,:], vmin=-1, vmax=1, cmap=sns.color_palette("hls", 256)).set_title('Attention');
fig = plt.figure(figsize=(18,2))
sns.heatmap(x[0,:,:], vmin=Xt_min, vmax=Xt_max, cmap=sns.color_palette("hls", 256)).set_title('Init Features Data');
for i in range(0,BATCH_SIZE,8):
    fig = plt.figure(figsize=(18,2))
    sns.heatmap(encEmb[i,:,:], vmin=-10, vmax=10, cmap=sns.color_palette("hls", 256)).set_title('Embedding');
for i in range(0,BATCH_SIZE,8):
    fig = plt.figure(figsize=(18,2))
    sns.heatmap(decEmb[i,:,:], vmin=-10, vmax=10, cmap=sns.color_palette("hls", 256)).set_title('Embedding');
# for i in range(0,BATCH_SIZE,8):
#     fig = plt.figure(figsize=(18,2))
#     sns.heatmap(out[i,:,:], vmin=Out_min, vmax=Out_max, cmap=sns.color_palette("hls", 256)).set_title('Encoder');